# Lab 11: How to gate a RAG change before release

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls; the release tests need none) | In class**

## Goal

Keep the documents a user may not read out of that user's evidence, and check the filter by reading the replies, not
only the hits. The moment that teaches it: with no filter, a customer's question about reset links finds three
documents customers may not read, and every answer repeats the runbook's rule. Filtered inside the search, the
customer gets only public policies; yet asked how long a support agent can make a reset link last, the model answers
with Atlas's public 30 minutes, in our runs often as if a support agent sets it. **The hits were right, and the reply
was still wrong.**

Parts C to F (self-study) stop a change before release with tests that read its saved results: the change passes more
questions than today's release, and the tests still fail it, because a note customers may not read reached their
evidence. No reply repeated the note, so only a check on the evidence catches it.

Your pipeline now answers follow-ups, and you can measure any change question by question, but every user can read
everything in the index, and nothing stops a change that scores better while breaking a rule. This lab adds
documents customers may not read, and writes the tests a change must pass before release.

## What you will learn

- How to label every document with who may read it, and list what each user may read by hand.
- How to make one filter per user with a function that returns a function, and use it after the search and inside it.
- Why a permission filter is checked by reading the replies, not only the hits.
- How to find an outdated version of a policy in the index and remove it by its ID.
- How to save what a change did, one row per question, with a record of what produced the run.
- How to write release checks as pytest tests, run them on saved results with no key, read a failing assertion, and
  say why a better average cannot pass a broken rule.

**In class:** Parts A and B. Parts C to F are self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-27
OPENAI_API_KEY    found in .env


This lab uses `chat_model`, `embed_model` and `JSON_MODE`. It makes 148 short chat calls and embeds the corpus once
(the in-class Parts A and B make 13 of the calls). The saved outputs and every count in the text come from
`gpt-4o-mini`; on another model the failures can move to other questions. Your own run will differ a little: model replies vary from run
to run and similarity scores can move in the third decimal, so compare the order of results and the kind of reply with
the text, not every number.

---

## Part A: Label who may read each document

Part A adds documents that not every user may read, and gives every document in the index a label that says who may.

The support team wants its agents to use the assistant too, with their own documents: three pages of the support
runbook and one note from the incident team. Customers must never read them. They are in
`resources/constructed-records.json`, written for this lab; the five policies in `data/support-case.json` are
unchanged. Load the file and print the four restricted records:

In [3]:
# constructed: the lab's records file; its "restricted" list holds the four documents customers may not read
import json

LAB = ROOT / "11_Release_Gate"
constructed = json.loads((LAB / "resources" / "constructed-records.json").read_text(encoding="utf-8"))
for record in constructed["restricted"]:
    print(record["chunk_id"], f"{record['access']:8}|", record["text"])

R01 staff   | Support agents can send a reset link that stays valid for 4 hours to an Atlas workspace owner who is locked out. Do not offer it to other users.
R02 staff   | Support agents reset temporary Atlas accounts from the admin console after they confirm the account's end date.
R03 staff   | Support agents can export all reports of an Atlas workspace when its owner asks in writing.
R04 internal| When a customer says an Atlas reset link expired too soon, check the request time in the admin console before you send a new link.


Each line is one record: its ID, its label and its text. The label is an **access label**, the group of readers the
document is meant for: `staff` on the three runbook pages (R01 to R03), `internal` on the incident note (R04), because
each team labels its own documents. Checking a document's label against the user before the model sees the document
is **access control**; in RAG systems it is also called **permission filtering**.

The same file says which labels each kind of user may read:

In [4]:
# USERS: for each kind of user, the labels that user may read
USERS = constructed["users"]
USERS

{'customer': ['public'], 'agent': ['public', 'staff', 'internal']}

`USERS` is a dictionary with one key per kind of user:

- `customer` may read `public` only.
- `agent` may read `public`, `staff` and `internal`.

A list of what a user may read is an **allow-list**: a label nobody added to it is read by nobody, so a new label stays
hidden until someone decides who may read it.

The corpus comes from `shared/lc_pipeline.py`, the pipeline Lab 06 Part D shows: `build_store()` embeds documents into
an in-memory vector store, `retrieve()` returns the three closest with their scores, `evidence()` writes them as
`[ID] text` lines, and `answer()` asks the model for an answer that cites those lines. Two more functions feed it:
`load_corpus()` reads the 277 records (the 272 chunks and the five policies), and `to_documents()` wraps each record
as a LangChain `Document`.

The 272 chunks carry no label at all; only the policies do. Give every record a label when the index is built, so a
filter never has to guess. Try it on one chunk first:

In [5]:
# chunk: the first record of the corpus; labelled: the same record with an access label added
from shared import lc_pipeline

chunk = lc_pipeline.load_corpus()[0]
labelled = {"access": "public", **chunk}
print("before:", list(chunk))
print("after: ", list(labelled))

before: ['chunk_id', 'source_id', 'title', 'start', 'end', 'text', 'tokens']
after:  ['access', 'chunk_id', 'source_id', 'title', 'start', 'end', 'text', 'tokens']


`{"access": "public", **chunk}` builds a new dictionary: first `access` set to `public`, then every field of `chunk`
(`**` copies a dictionary's fields into the new one). When the record already has an `access` field, it comes later
and wins, so a policy keeps its own label. Now label all 277 records and add the four restricted ones:

In [6]:
# records: the 277 corpus records, each with an access label, then the four restricted records
from collections import Counter

records = [{"access": "public", **record} for record in lc_pipeline.load_corpus()] + constructed["restricted"]
print(len(records), "records |", Counter(record["access"] for record in records))

281 records | Counter({'public': 277, 'staff': 3, 'internal': 1})


281 records: 277 from the corpus plus the four restricted ones. `Counter` counts how often each value appears: 277
`public`, 3 `staff`, 1 `internal`.

Before any filter, list by hand what each user may read:

In [7]:
# allowed: the IDs of the records whose label is in this user's list
for user, labels in USERS.items():
    allowed = [record["chunk_id"] for record in records if record["access"] in labels]
    print(f"{user:9} may read {len(allowed)} of {len(records)} records")

customer  may read 277 of 281 records
agent     may read 281 of 281 records


A customer may read 277 of the 281, every record but the four restricted ones; an agent may read all 281. These counts
are the answer key for every filter in Part B.

Build the store with `build_store()`. `to_documents()` keeps each record's text as the document's content and every
other field, the label included, as its **metadata**: the fields a filter can read.

In [8]:
# store: the 281 records embedded into an InMemoryVectorStore; then R04's metadata, label included
store = lc_pipeline.build_store(lc_pipeline.to_documents(records), settings)
print(len(store.store), "documents")
print(store.get_by_ids(["R04"])[0].metadata)

281 documents
{'chunk_id': 'R04', 'source_id': 'R04', 'title': 'Atlas incident note: expired links', 'version': '2026-10-01', 'access': 'internal'}


`store.store` is the dictionary in which `InMemoryVectorStore` keeps each document by its ID; this lab reads it to
count and list everything in the index (a store backed by a database is listed with a query instead). `get_by_ids()`
returns the documents with those IDs. R04's metadata holds `access: internal`: the label is in the index, where a
filter can read it.

**Lesson to Remember**
> Label every document when the index is built, and keep the list of what each user may read in one place. A filter
> can only apply a label that is there.

### Checkpoint A

| You should see | Our run |
|---|---|
| The restricted records | R01 to R03 labelled `staff`, R04 labelled `internal` |
| `USERS` | customer: `public`; agent: `public`, `staff`, `internal` |
| Labels | 281 records: 277 `public`, 3 `staff`, 1 `internal` |
| Allowed | customer 277 of 281, agent 281 of 281 |
| The store | 281 documents; R04's metadata holds `access: internal` |

---

## Part B: Filter by permission after the search, then inside it

Part B keeps restricted documents out of a customer's evidence, in the two places a filter can sit.

A customer asks:

> "Can support send me a new Atlas reset link?"

*Predict first:* with no filter, how many of the three hits will be documents this customer may not read?

In [9]:
# hits: the three closest documents for the customer's question, with no filter (score, ID, label, start of the text)
question = "Can support send me a new Atlas reset link?"
hits = lc_pipeline.retrieve(store, question)
for document, score in hits:
    print(f"{score:.3f} {document.id} {document.metadata['access']:8}| {document.page_content[:70]}")

0.638 R01 staff   | Support agents can send a reset link that stays valid for 4 hours to a
0.634 R04 internal| When a customer says an Atlas reset link expired too soon, check the r
0.561 R02 staff   | Support agents reset temporary Atlas accounts from the admin console a


Each hit is a pair: a LangChain `Document` (its `id`, `page_content` and `metadata`) and its cosine score, best first.
All three are documents the customer may not read: two runbook pages and the incident note. They rank first because
they are about exactly this, support sending reset links. Answer from them three times:

In [10]:
# three answers from the unfiltered hits; .answer is the text of the cited reply
for _ in range(3):
    print(lc_pipeline.answer(question, lc_pipeline.evidence(hits), settings).answer)

Yes, support can send you a new Atlas reset link if you are an Atlas workspace owner who is locked out.


Yes, support can send you a new Atlas reset link if you are an Atlas workspace owner who is locked out.


Yes, support can send you a new Atlas reset link if you are an Atlas workspace owner who is locked out.


All three replies answer *Yes* and add the runbook's condition, that the customer owns the workspace and is locked
out. The customer has just read a runbook rule. No rule in the prompt can undo that: the text already reached the
model, and every log of the request. This is the leak OWASP calls **sensitive information disclosure** (Resources).

### One filter per user

A **filter** here is a function that takes one document and returns `True` to keep it or `False` to drop it. Which
documents it keeps depends on who asks: the customer's filter drops `staff` and `internal`, the agent's keeps them. So
write one function that takes a user and *returns* that user's filter:

In [11]:
# can_read(user) makes the filter for one user: a function that is True for a document whose label the user may read
def can_read(user):
    """A filter for one user: True for a document whose access label the user may read."""
    def allowed(document):
        return document.metadata["access"] in USERS[user]
    return allowed

Read it from the inside out:

- `allowed(document)` is the filter: it looks up the document's label in `USERS[user]`.
- `can_read(user)` defines `allowed` and returns it, without calling it.

Call `can_read` for the customer and print what comes back:

In [12]:
# customer_filter: what can_read returns for the customer, a function not yet called on any document
customer_filter = can_read("customer")
print(customer_filter)

<function can_read.<locals>.allowed at 0x11968f240>


`can_read` returned a function, `allowed`, made inside `can_read` (that is what `<locals>` says); the hex number is
where it sits in memory and changes on every run. Now call it on two documents, the runbook page R01 (`staff`) and
today's Atlas expiry policy H02 (`public`), and do the same with the agent's filter:

In [13]:
# each user's filter called on one restricted document (R01) and one public document (H02)
r01 = store.get_by_ids(["R01"])[0]
h02 = store.get_by_ids(["H02"])[0]
print("customer:", customer_filter(r01), customer_filter(h02))
print("agent:   ", can_read("agent")(r01), can_read("agent")(h02))

customer: False True
agent:    True True


The customer's filter drops R01 and keeps H02; the agent's keeps both. `can_read("agent")(r01)` does both steps in
one line: `can_read("agent")` makes the agent's filter, and `(r01)` calls it on R01. `allowed` still knows which user it belongs to
after `can_read` has returned, because it keeps the `user` it was made with. A function that keeps the variables of
the function that made it is a **closure**.

Why the search wants a function: the store calls the filter on each document, with the document as its only argument,
and knows nothing about users. The closure carries the user in, so one `can_read` makes the filter for every user.

### Filter after the search

The first place for a filter is after the search: search as before, then drop the hits the user may not read. This is
a **post-filter**. Apply the customer's filter to the three hits:

In [14]:
# post: the unfiltered hits that the customer's filter keeps
post = []
for document, score in hits:
    if customer_filter(document):
        post.append((document, score))
print("post-filter:", [document.id for document, _ in post])

post-filter: []


Nothing is left: all three places went to documents the customer may not read. Answer from what is left, no evidence
at all:

In [15]:
# one answer from the post-filtered hits: an empty evidence list
print(lc_pipeline.answer(question, lc_pipeline.evidence(post), settings).answer)

Insufficient evidence.


*Insufficient evidence.*, from nothing. How often does a post-filter leave fewer than three hits? Count it on Lab 06's
golden set: 38 questions with their accepted answers (the 30 development questions and eight Atlas and Boreal policy
cases). Searches only, no chat calls:

In [16]:
# kept: how many golden questions end with 0, 1, 2 or 3 hits after the customer's post-filter
from shared import evals

golden = evals.golden_set()
kept = Counter()
for record in golden:
    left = 0
    for document, _ in lc_pipeline.retrieve(store, record["question"]):
        if customer_filter(document):
            left += 1
    kept[left] += 1
print("hits left of 3, and how many questions:", dict(sorted(kept.items())))

hits left of 3, and how many questions: {1: 5, 2: 2, 3: 31}


### What happens step by step

1. `evals.golden_set()` loads the 38 golden records; each has a `question`.
2. For each question, `retrieve()` returns the three closest documents, with no filter.
3. `left` counts the hits the customer's filter keeps.
4. `kept[left] += 1` adds the question to the count for that number of hits; a `Counter` starts every count at 0.
5. `dict(sorted(...))` prints the counts in order of hits left.

The post-filter keeps all three hits for 31 of the 38 questions, two hits for 2 questions, and only one hit
for 5. Each restricted
document that ranks high takes a place a public one could have had, and the customer gets less evidence.

### Filter inside the search

The second place is inside the search: give the filter to the store, which drops the documents the user may not read
before it ranks the rest. This is a **pre-filter**; vector databases call it a metadata filter on the search.
`retrieve()` hands its `filter` to the store:

In [17]:
# retrieve() from shared/lc_pipeline.py: the filter goes to the store's own search
import inspect

print(inspect.getsource(lc_pipeline.retrieve))

def retrieve(store: InMemoryVectorStore, question: str, k: int = 3, filter=None) -> list:
    """The k closest documents, best first, as (Document, cosine score) pairs; filter is a function of a Document."""
    return store.similarity_search_with_score(question, k=k, filter=filter)



`similarity_search_with_score()` calls `filter(document)` on every document in the store, keeps those that return
`True`, and ranks only them by cosine score. Search with the customer's filter:

In [18]:
# pre: the three closest documents among those the customer may read
pre = lc_pipeline.retrieve(store, question, filter=customer_filter)
print("pre-filter:", [document.id for document, _ in pre])

pre-filter: ['H02', 'H01', 'H04']


Three hits again, all `public`. Answer from them three times:

In [19]:
# three answers from the pre-filtered hits
for _ in range(3):
    print(lc_pipeline.answer(question, lc_pipeline.evidence(pre), settings).answer)

Insufficient evidence.


Insufficient evidence.


Insufficient evidence.


The pre-filter fills all three places with documents the customer may read: H02, H01 and H04. All three answers are
*Insufficient evidence.* No public policy says that support sends links (H02 tells the customer to request a new one),
so refusing is right here.

- A post-filter can leave fewer than *k* hits, down to none.
- A pre-filter returns *k* hits whenever the user may read at least *k* documents.

**Filter inside the search, so that a user's evidence is the best of what that user may read.**

### Check what the filter lets through

A filter is a guard, and a guard is measured on what it should let through as well as on what it should stop. Ask a
question that only the runbook answers, once as an agent and once as a customer:

> "How long can a support agent make an Atlas reset link last?"

*Predict first:* the customer's filter removes R01, the page that answers. What will the customer's answers say?

In [20]:
# for each user: the hits their filter allows for the runbook question, then three answers from those hits
staff_question = "How long can a support agent make an Atlas reset link last?"
for user in USERS:
    user_hits = lc_pipeline.retrieve(store, staff_question, filter=can_read(user))
    print(user, [document.id for document, _ in user_hits])
    for _ in range(3):
        print("  ", lc_pipeline.answer(staff_question, lc_pipeline.evidence(user_hits), settings).answer)

customer ['H02', 'H01', 'H04']


   A support agent can make an Atlas reset link last for 30 minutes.


   A support agent can make an Atlas reset link last for 30 minutes.


   A support agent can make an Atlas reset link last for 30 minutes.


agent ['R04', 'R01', 'H02']


   Support agents can make an Atlas reset link last for 4 hours.


   Support agents can make an Atlas reset link last for 4 hours.


   Support agents can make an Atlas reset link last for 4 hours.


Two results:

- The agent gets R04, R01 and H02, and *4 hours*, R01's rule, in 3 of 3: the filter lets through what it should.
- The customer gets H02, H01 and H04, and *A support agent can make an Atlas reset link last for 30 minutes.* in 3
  of 3. In another run of this notebook it said so once in three; the other two replies said *Atlas reset links
  expire after 30 minutes.*

H02's 30 minutes is about every reset link; nothing in it is about agents. The filter did its job, and the model then
answered from the nearest document the customer may read, as if it were about agents. The hits were right and the
replies were still wrong.

**Check a permission filter by reading the replies, not only the hits.**

**Lesson to Remember**
> A filter for one user is a function the search calls on every document; `can_read(user)` makes it, and the closure
> keeps the user. Put the filter inside the search, so the user gets the best documents they may read, then read the
> replies: a correct filter can still leave the model answering from the wrong document.

### Checkpoint B

| You should see | Our run |
|---|---|
| No filter | R01, R04, R02; *Yes* with the runbook's condition, 3 of 3 |
| The filters | `customer: False True`, `agent: True True` |
| Post-filter | no hits; *Insufficient evidence.*; on the 38 golden questions three hits left for 31, two for 2, one for 5 |
| Pre-filter | H02, H01, H04; *Insufficient evidence.* 3 of 3 |
| The runbook question | agent: R04, R01, H02, *4 hours* 3 of 3; customer: H02, H01, H04, *... last for 30 minutes.* 3 of 3 (1 to 3 of 3 across our runs) |

---

## Part C: Replace an outdated policy in the index (self-study)

Part C keeps old versions out of the evidence: a filter by permission cannot see that a document is out of date.

Until 1 October, H02 said that Atlas reset links last 24 hours. Suppose the index was first built in April, from an
export that named each document by its source and its date (`H02-2026-04-01`), and that the October update added
today's H02 beside it. Keeping track of which version of each document is current is **document versioning**. Add the
April document to the store:

In [21]:
# april: the April version of H02, with its own ID; add_documents() embeds it and adds it to the store
april = constructed["outdated"]
print(april["chunk_id"], april["version"], "|", april["text"])
store.add_documents(lc_pipeline.to_documents([april]))
print(len(store.store), "documents")

H02-2026-04-01 2026-04-01 | Atlas reset links expire after 24 hours. Request a new link if the old link has expired.


282 documents


282 documents: the April H02 sits beside today's. Ask G01 and G02, the golden set's two questions about how long a
link lasts, as a customer, three times each:

In [22]:
# for each expiry question: the customer's hits, then three answers from them
expiry = ["How long does an Atlas reset link remain valid?",
          "When does the password recovery email stop working for Atlas?"]
for q in expiry:
    q_hits = lc_pipeline.retrieve(store, q, filter=can_read("customer"))
    print(q, [document.id for document, _ in q_hits])
    for _ in range(3):
        print("  ", lc_pipeline.answer(q, lc_pipeline.evidence(q_hits), settings).answer)

How long does an Atlas reset link remain valid? ['H02', 'H02-2026-04-01', 'H04']


   Insufficient evidence.


   Insufficient evidence.


   Insufficient evidence.


When does the password recovery email stop working for Atlas? ['H01', 'H04', 'H02-2026-04-01']


   The password recovery email for Atlas stops working after 24 hours.


   The password recovery email for Atlas stops working after 24 hours.


   The password recovery email for Atlas stops working after 24 hours.


Two results:

- The first question gets both versions, H02 and the April document, and the model refuses in 3 of 3: its evidence
  contradicts itself.
- The second gets H01, H04 and the April document; today's H02 ranks fourth, outside the three, and all three answers
  say *The password recovery email for Atlas stops working after 24 hours.* The answer is faithful to its evidence,
  and wrong under today's policy.

The update did what it was asked: adding a document replaces only a document with the same ID, and the April one has
another. Both are in the store:

In [23]:
# both versions of H02, read back from the store by ID
for document in store.get_by_ids(["H02", "H02-2026-04-01"]):
    print(document.id, document.metadata["version"], "|", document.page_content)

H02 2026-10-01 | Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
H02-2026-04-01 2026-04-01 | Atlas reset links expire after 24 hours. Request a new link if the old link has expired.


To find every outdated document, compare each document's version with the version its owner publishes today, in the
policy file or the records file. First the current versions:

In [24]:
# CURRENT: for each source ID, the version its owner publishes today (the five policies and the four restricted records)
policies = json.loads((ROOT / "data" / "support-case.json").read_text(encoding="utf-8"))
CURRENT = {doc["source_id"]: doc["version"] for doc in policies + constructed["restricted"]}
CURRENT

{'H01': '2026-10-01',
 'H02': '2026-10-01',
 'H03': '2026-10-01',
 'H04': '2026-10-01',
 'H05': '2026-10-01',
 'R01': '2026-10-01',
 'R02': '2026-10-01',
 'R03': '2026-10-01',
 'R04': '2026-10-01'}

Nine sources, each with today's version. A document whose version is not its source's current version is
**stale**. The chunks have no version, so the check skips them:

In [25]:
# outdated(store): the IDs of documents whose version is not the current version of their source
def outdated(store):
    """The IDs of documents whose version is not the current version of their source."""
    stale = []
    for doc in store.store.values():
        version = doc["metadata"].get("version")
        if version and version != CURRENT[doc["metadata"]["source_id"]]:
            stale.append(doc["id"])
    return stale

print("outdated:", outdated(store))

outdated: ['H02-2026-04-01']


### What happens step by step

1. `store.store.values()` gives every document as a dictionary with `id`, `text` and `metadata` (a different shape
   from the `Document` objects `retrieve()` returns).
2. `.get("version")` returns `None` for a chunk, which has no version, and the `if` skips it.
3. A document whose version differs from its source's current version goes in the list.

One stale document, the April H02. Delete it by its ID, and check that it is gone:

In [26]:
# delete the April document by ID; then what get_by_ids finds, what outdated() finds, and the document count
store.delete(["H02-2026-04-01"])
print("found:", [document.id for document in store.get_by_ids(["H02", "H02-2026-04-01"])],
      "| outdated:", outdated(store), "|", len(store.store), "documents")

found: ['H02'] | outdated: [] | 281 documents


Only today's H02 is found, nothing is outdated, and the store is back to 281 documents. Ask the second question again:

In [27]:
# the second expiry question again, as a customer: hits, then three answers
q_hits = lc_pipeline.retrieve(store, expiry[1], filter=can_read("customer"))
print([document.id for document, _ in q_hits])
for _ in range(3):
    print("  ", lc_pipeline.answer(expiry[1], lc_pipeline.evidence(q_hits), settings).answer)

['H01', 'H04', 'H02']


   The password recovery email for Atlas stops working after 30 minutes.


   The password recovery email for Atlas stops working after 30 minutes.


   The password recovery email for Atlas stops working after 30 minutes.


Today's H02 is third, and all three answers say *30 minutes*. Deleting by ID works when you know the old ID. When an
update keeps the ID, `add_documents` replaces the old document in place. Add today's H02 again:

In [28]:
# h02: today's H02 record; adding it again under the same ID replaces the stored document
h02 = [record for record in records if record["chunk_id"] == "H02"]
store.add_documents(lc_pipeline.to_documents(h02))
print(len(store.store), "documents | H02 version", store.get_by_ids(["H02"])[0].metadata["version"])

281 documents | H02 version 2026-10-01


Still 281 documents: the same ID replaced the document instead of adding one. So:

- give each document an ID that stays the same across its versions, and an update replaces it in place;
- when an ID changes, delete the old one in the same update.

`outdated()` becomes a release test in Part E, so a change cannot ship with an old version in its index.

**Lesson to Remember**
> A permission filter cannot see age. An update that adds a new version under a new ID leaves the old one answering;
> check every document's version against its source, and delete the old ID in the same update.

### Checkpoint C

| You should see | Our run |
|---|---|
| April's H02 added | 282 documents |
| Both versions in the store | first question: H02, H02-2026-04-01, H04, *Insufficient evidence.* 3 of 3; second: H01, H04, H02-2026-04-01, *24 hours* 3 of 3 |
| The version check | `outdated: ['H02-2026-04-01']`, then `[]` after `delete`, 281 documents |
| After the delete | H01, H04, H02; *30 minutes* 3 of 3 |
| Same ID added again | 281 documents, H02 version 2026-10-01 |

---

## Part D: Save the rows of a change, with its manifest (self-study)

Part D runs today's release and the change on the same questions and saves what each one did, so tests can read it
later without the store or a key.

The **release set** is the fixed list of questions every change runs before release: Lab 06's 38 golden records, asked
by a customer, plus the four cases in the records file: three agent questions that the runbook or the note answers,
and Part B's customer question, which should be refused.

In [29]:
# release_set: each golden record with user "customer" added, then the four cases of the records file
release_set = [{**record, "user": "customer"} for record in golden] + constructed["cases"]
print(len(release_set), "questions |", Counter(record["user"] for record in release_set))
for record in constructed["cases"]:
    print(record["question_id"], record["user"], record["references"] or "refuse", "|", record["question"])

42 questions | Counter({'customer': 39, 'agent': 3})
A01 agent ['4 hours'] | How long can a support agent make an Atlas reset link last?
A02 agent ['admin console'] | How do support agents reset a temporary Atlas account?
A03 agent ['request time'] | What should I check when a customer says an Atlas reset link expired too soon?
A04 customer refuse | Can support send me a new Atlas reset link?


42 questions: 39 asked by a customer, 3 by an agent. A01 to A03 carry the words a right answer must hold; A04 must be
refused.

One **row** per question holds what a test needs: who asked, the IDs packed (the documents the model read), the
answer, whether it passed, the claims and the evidence lines. It is Lab 06's `evals.score()` plus the user and a
filter for that user, which `score()` does not take. `filter_for` is a function like `can_read`: give it a user and it
returns that user's filter.

In [30]:
# row(record, filter_for): one question through the pipeline with its user's filter, as the dictionary the tests read
def row(record, filter_for):
    """One question through the pipeline, searched with filter_for(user); a record with a pack uses those documents."""
    if "pack" in record:
        hits = [(document, None) for document in store.get_by_ids(record["pack"])]
    else:
        hits = lc_pipeline.retrieve(store, record["question"], filter=filter_for(record["user"]))
    lines = lc_pipeline.evidence(hits)
    reply = lc_pipeline.answer(record["question"], lines, settings)
    refused = reply.answer.strip() == evals.REFUSAL
    passed = refused if record["refuse"] else evals.found(reply.answer, record["references"])
    return {"question_id": record["question_id"], "user": record["user"], "type": record["type"],
            "refuse": record["refuse"], "packed": [document.id for document, _ in hits], "answer": reply.answer,
            "refused": refused, "passed": passed, "claims": [claim.model_dump() for claim in reply.claims],
            "evidence": lines}

### What happens step by step

1. A record with a `pack` (Lab 06's G03) answers from those fixed documents; every other record is searched with its
   user's filter.
2. `evidence()` writes the hits as lines, and `answer()` makes one chat call.
3. `refused` is `True` when the answer is exactly *Insufficient evidence.*
4. A record that should be refused passes when the model refused; any other passes when one of its accepted answers is
   in the reply (Lab 06's `found()`).
5. The row keeps the fields the tests read; `model_dump()` turns each claim into a plain dictionary that JSON can save.

A row says what happened; the **build manifest** says what produced it: every document in the index with its source,
version, access label and a short hash of its text, then the search settings, a hash of the answer rules, the release
set's version (a hash of its 42 records, from Lab 06's `version()`) and the stamp. A **hash** is a short fingerprint of a text: change one character
and the hash changes. With the manifest, a test can check the index a run used, and anyone can rebuild that run later:

In [31]:
# manifest(store, search): what produced a run, the documents in the index and the settings
import hashlib
from shared.answer import RULES

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

def manifest(store, search):
    documents = [{"id": doc["id"], "source_id": doc["metadata"]["source_id"], "version": doc["metadata"].get("version"),
                  "access": doc["metadata"]["access"], "text": sha(doc["text"])} for doc in store.store.values()]
    return {"search": search, "k": 3, "rules": sha(RULES), "questions": evals.version(release_set, settings)["records"],
            **stamp(settings), "documents": documents}

print({name: value for name, value in manifest(store, "example").items() if name != "documents"})

{'search': 'example', 'k': 3, 'rules': '4be8e10f41b6', 'questions': '3b9ff274df1a', 'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-27'}


### What happens step by step

1. `sha()` hashes a text with SHA-256 and keeps the first 12 hexadecimal characters.
2. `documents` lists every document in the store with the four fields the tests read and the hash of its text.
3. The rest records the settings: the search (a description), `k`, the rules' hash, the release set's hash, and the
   models and endpoints from `stamp()`.
4. The print leaves out the 281 documents.

Today's release lets every user read what a customer may read. The change gives agents the runbook and the note; its
filter came from a teammate, and it hides from customers every document labelled `staff`. Both are written like
`can_read`: a function of the user that returns a filter.

In [32]:
# today(user): today's release, the customer's filter for every user; hide_staff(user): the change's filter
def today(user):
    return can_read("customer")

def hide_staff(user):
    """The change's filter: a customer never sees a document labelled staff."""
    def allowed(document):
        return user != "customer" or document.metadata["access"] != "staff"
    return allowed

`hide_staff` is a **deny-list**: it names what to hide and lets everything else through. `can_read` is the allow-list
of Part A: it names what to show.

*Predict first:* does `hide_staff` keep every restricted record away from a customer?

Run both on the 42 questions (84 chat calls) and save each run as a **results file**: the manifest and the rows, in
`results/`:

In [33]:
# run(name, filter_for, search): one row per release question, saved with the manifest as results/<name>.json
from shared.settings import chat_url

RESULTS = LAB / "results"

def run(name, filter_for, search):
    rows = [row(record, filter_for) for record in release_set]
    results = {"manifest": manifest(store, search), "rows": rows}
    (RESULTS / f"{name}.json").write_text(json.dumps(results, indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"{name:9} passed {sum(r['passed'] for r in rows)} of {len(rows)} | saved results/{name}.json")
    return rows

baseline = run("baseline", today, "every user reads public documents, k=3")
candidate = run("candidate", hide_staff, "customers: no staff documents; agents: all, k=3")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

baseline  passed 35 of 42 | saved results/baseline.json


candidate passed 37 of 42 | saved results/candidate.json

gpt-4o-mini at https://api.openai.com/v1


### What happens step by step

1. `run()` makes one row per release question with the given filter.
2. It saves the manifest and the rows as JSON in `results/<name>.json`.
3. It prints how many rows passed and returns the rows.
4. The last line names the model and endpoint behind the numbers.

The change passes 37 of 42, today's release 35. Which questions moved? Lab 08's `paired()` compares two runs question
by question: `both` counts the questions that passed in both runs, `neither` those that failed in both, and `gained`
and `lost` list the positions that changed.

In [34]:
# change: the two runs compared question by question; the lost questions printed with the change's answer
from shared.hybrid import paired

change = paired([r["passed"] for r in baseline], [r["passed"] for r in candidate])
print("both passed", change["both"], "| both failed", change["neither"])
print("gained:", [baseline[i]["question_id"] for i in change["gained"]])
print("lost:  ", [(baseline[i]["question_id"], candidate[i]["answer"]) for i in change["lost"]])

both passed 34 | both failed 4
gained: ['A01', 'A02', 'A03']
lost:   [('G02', 'Insufficient evidence.')]


34 questions pass in both runs and 4 fail in both. The change gains the three agent questions, A01 to A03, which
today's release cannot answer, and loses one, G02, now *Insufficient evidence.* A question that passed before a change
and fails after it is a **regression**. On its average, the change is better. Part E asks whether it may ship.

**Lesson to Remember**
> A results file holds what a run did (one row per question) and what produced it (the manifest). Tests read the file,
> not the store, so they can run anywhere, later, with no key.

### Checkpoint D

| You should see | Our run |
|---|---|
| The release set | 42 questions: 39 customer, 3 agent |
| Saved | `results/baseline.json`, `results/candidate.json` |
| Passed | today's release 35 of 42, the change 37 |
| Paired | both passed 34, both failed 4; gained A01, A02, A03; lost G02 |

---

## Part E: Write the release tests and run them with pytest (self-study)

Part E turns the boundaries into tests that read the saved files, and runs them on the change. A **release gate** is
that set of checks: a change ships only when every one passes. A **regression test** checks that a change breaks
nothing that worked before, by comparing it with today's release on the same questions; two of the tests below are
regression tests.

Start with the access check by hand, on the saved results file: is every packed document one the row's user may read?
The labels come from the file's manifest, so the check reads the index the run used, not today's store:

In [35]:
# saved: the change's results file; access: each document's label from its manifest; then every pack that breaks a rule
saved = json.loads((RESULTS / "candidate.json").read_text(encoding="utf-8"))
access = {doc["id"]: doc["access"] for doc in saved["manifest"]["documents"]}
for r in saved["rows"]:
    for doc_id in r["packed"]:
        if access[doc_id] not in USERS[r["user"]]:
            print(f"{r['question_id']} ({r['user']}) read {doc_id} ({access[doc_id]}) | {r['answer']}")

G01 (customer) read R04 (internal) | Atlas reset links expire after 30 minutes.
G02 (customer) read R04 (internal) | Insufficient evidence.
G05 (customer) read R04 (internal) | No, temporary Atlas accounts cannot use the usual email route.
G07 (customer) read R04 (internal) | Insufficient evidence.
G09 (customer) read R04 (internal) | Insufficient evidence.
A04 (customer) read R04 (internal) | Insufficient evidence.


Six customer packs hold R04, the incident note: G01, G02, G05, G07, G09 and A04.

- `hide_staff` names the label `staff`; R04 carries `internal`, a label the teammate never named. A deny-list lets
  through every label it does not list.
- An allow-list such as `can_read` fails the other way: a label it does not list stays hidden until someone adds it.

Read the replies: *Atlas reset links expire after 30 minutes.*, *No, temporary Atlas accounts cannot use the usual
email route.* and four *Insufficient evidence.* None repeats the note's request time or admin console, so a check on
the replies would pass this change. The note still went to the model, and into any trace or log of those requests:
**only a check on the packs sees it**.

**pytest** runs every function whose name starts with `test_` in a file whose name starts with `test_`, and reports
each `assert` that fails with its message ([pytest docs](https://docs.pytest.org/en/stable/getting-started.html)).
The release tests are in `test_release.py` in this folder. Read the first one:

In [36]:
# the first release test, printed from test_release.py
sys.path.insert(0, str(LAB))
import test_release

print(inspect.getsource(test_release.test_no_restricted_source_in_pack))

def test_no_restricted_source_in_pack(candidate):
    access = {doc["id"]: doc["access"] for doc in candidate["manifest"]["documents"]}
    leaks = [f"{row['question_id']} ({row['user']}) read {doc_id} ({access[doc_id]})"
             for row in candidate["rows"] for doc_id in row["packed"] if access[doc_id] not in USERS[row["user"]]]
    assert not leaks, f"{len(leaks)} packed documents the user may not read: " + "; ".join(leaks)



It is the check above, with the rows and the manifest read from a results file, and the packs that break the rule
collected in `leaks` instead of printed. The argument `candidate` is a **fixture**: pytest calls the function of that
name in the file, which loads the file named by `GATE_CANDIDATE` (`results/candidate.json` when it is not set), and
passes in what it returns. `USERS` is read from the records file, so the gate holds the permission rules, not the
change.

Run the tests as a **CI job** would (continuous integration: a server that runs the tests on every change): in a new
Python process, on the saved file, with the key set to empty:

In [37]:
# gate(candidate_file, *options): pytest on the release tests in a new process, on one results file, with no key
import os
import subprocess

def gate(candidate_file, *options):
    """Run the release tests on a saved results file, in a new process, as a CI job would."""
    command = [sys.executable, "-m", "pytest", "11_Release_Gate/test_release.py", "-q", "--tb=short", "--color=no", *options]
    env = {**os.environ, "GATE_CANDIDATE": str(candidate_file), "OPENAI_API_KEY": ""}   # no key needed
    print(subprocess.run(command, cwd=ROOT, env=env, capture_output=True, text=True).stdout)

gate(RESULTS / "candidate.json", "-k", "restricted")

F                                                                        [100%]
=================================== FAILURES ===================================
______________________ test_no_restricted_source_in_pack _______________________
11_Release_Gate/test_release.py:50: in test_no_restricted_source_in_pack
    assert not leaks, f"{len(leaks)} packed documents the user may not read: " + "; ".join(leaks)
E   AssertionError: 6 packed documents the user may not read: G01 (customer) read R04 (internal); G02 (customer) read R04 (internal); G05 (customer) read R04 (internal); G07 (customer) read R04 (internal); G09 (customer) read R04 (internal); A04 (customer) read R04 (internal)
E   assert not ['G01 (customer) read R04 (internal)', 'G02 (customer) read R04 (internal)', 'G05 (customer) read R04 (internal)', 'G07 (customer) read R04 (internal)', 'G09 (customer) read R04 (internal)', 'A04 (customer) read R04 (internal)']
=========================== short test summary info ==============

### What happens step by step

1. `command` is the command line a CI job would type: this environment's Python running pytest on the test file,
   quietly (`-q`), with each failure cut to its assertion (`--tb=short`) and no colour codes (`--color=no`).
2. `env` copies the environment, names the results file in `GATE_CANDIDATE`, and empties the key: the tests must not
   need one.
3. `subprocess.run()` starts the command as a new process from the repository folder (`cwd=ROOT`), waits for it, and
   keeps what it printed (`capture_output=True`, as text).
4. `-k restricted` runs only the tests whose names contain *restricted*.

`1 failed, 4 deselected`: the access test failed, and the line after `AssertionError` names each pack that broke it.
The other four tests:

In [38]:
# the other four release tests, printed from test_release.py
for test in (test_release.test_only_current_versions_in_index, test_release.test_refusal_slice_held,
             test_release.test_regressions_reviewed, test_release.test_quotes_found):
    print(inspect.getsource(test))

def test_only_current_versions_in_index(candidate):
    stale = [f"{doc['id']} ({doc['source_id']} {doc['version']}, current {CURRENT[doc['source_id']]})"
             for doc in candidate["manifest"]["documents"]
             if doc["version"] and doc["version"] != CURRENT[doc["source_id"]]]
    assert not stale, "outdated versions in the index: " + "; ".join(stale)

def test_refusal_slice_held(baseline, candidate):
    held = {row["question_id"] for row in baseline["rows"] if row["refuse"] and row["passed"]}
    broken = [f"{row['question_id']}: {row['answer']}" for row in candidate["rows"]
              if row["question_id"] in held and not row["passed"]]
    assert not broken, "refusals the baseline held, now answered: " + "; ".join(broken)

def test_regressions_reviewed(baseline, candidate):
    passed_before = {row["question_id"] for row in baseline["rows"] if row["passed"]}
    lost = [f"{row['question_id']}: {row['answer']}" for row in candidate["rows"]
            if row["ques

| Test | Fails when | Kind |
|---|---|---|
| `test_no_restricted_source_in_pack` | a pack holds a document its user may not read | hard: never allowed |
| `test_only_current_versions_in_index` | the index holds a document older than its source's current version (Part C) | hard |
| `test_refusal_slice_held` | the change answers a question the baseline rightly refused | hard; a regression test |
| `test_regressions_reviewed` | a question the baseline passed fails now, and nobody reviewed it | reviewed: a person may accept it; a regression test |
| `test_quotes_found` | the quotes-found rate is under 0.95 | threshold |

`test_quotes_found` is the one test with a number in it. It needs the change's **quotes-found rate** to reach
`MIN_QUOTES_FOUND`, set to 0.95 at the top of the file:

$$
\text{quotes-found rate} = \frac{\text{quotes found}}{\text{quotes checked}} \ge 0.95
$$

- quotes checked: every claim in every row of the change; each claim carries one quote.
- quotes found: the claims whose quote Lab 02's `quote_found()` finds, word for word, in the evidence line that the
  claim's source ID names.
- 0.95: the threshold, `MIN_QUOTES_FOUND`.

Compute it by hand on the change:

In [39]:
# checked and found: the same count test_quotes_found makes, on the change's saved rows
from shared.answer import quote_found

checked = 0
found = 0
for r in saved["rows"]:
    for claim in r["claims"]:
        checked += 1
        found += quote_found(claim, r["evidence"])
print(f"{found} of {checked} quotes found | rate {found / checked:.3f} | threshold {test_release.MIN_QUOTES_FOUND}")

37 of 37 quotes found | rate 1.000 | threshold 0.95


`found += quote_found(...)` adds 1 when the quote is found and 0 when it is not. All 37 of the change's quotes were found, so the rate is 1.000 and the test passes.

The 0.95 is not derived from the data: the team picks it before the run, so that no result can move it. What it allows depends on how many quotes a run has.
With the change's 37, one quote not found gives 36 of 37, about 0.973, and passes; two give 35 of 37, about 0.946, and fail. That is the intent: one quote that the word-for-word check misreads (a quote cut in an unusual way, a word the model changed) does not block a release, while several quotes the evidence does not hold do. A threshold of 1.0 would block a release over a single quote.

A reviewed regression goes in `REVIEWED`, the dictionary at the top of the file, with the reason, in a reviewed
commit: a loss can be accepted, but only by someone who read the row. Run all five tests:

In [40]:
# all five release tests on the change
gate(RESULTS / "candidate.json")

F..F.                                                                    [100%]
=================================== FAILURES ===================================
______________________ test_no_restricted_source_in_pack _______________________
11_Release_Gate/test_release.py:50: in test_no_restricted_source_in_pack
    assert not leaks, f"{len(leaks)} packed documents the user may not read: " + "; ".join(leaks)
E   AssertionError: 6 packed documents the user may not read: G01 (customer) read R04 (internal); G02 (customer) read R04 (internal); G05 (customer) read R04 (internal); G07 (customer) read R04 (internal); G09 (customer) read R04 (internal); A04 (customer) read R04 (internal)
E   assert not ['G01 (customer) read R04 (internal)', 'G02 (customer) read R04 (internal)', 'G05 (customer) read R04 (internal)', 'G07 (customer) read R04 (internal)', 'G09 (customer) read R04 (internal)', 'A04 (customer) read R04 (internal)']
__________________________ test_regressions_reviewed _____________

`2 failed, 3 passed`. The second failure is G02, which today's release answered: in the change's pack R04 took second
place, H02 dropped out of the three, and the model refused. Do not add G02 to `REVIEWED`: the cause is the leak, so the
fix belongs in the change, not in the gate.

**No test reads the average: a higher pass rate cannot buy back a document a customer should never have read.**

**Lesson to Remember**
> A release gate is a set of tests a change must pass before it ships. Hard rules (access, versions, refusals) fail on
> one row; a regression may pass only after a person reviews it; a threshold is set before the run. Run them on saved
> results, in a new process, with no key.

### Checkpoint E

| You should see | Our run |
|---|---|
| By hand | R04 in six customer packs: G01, G02, G05, G07, G09, A04 |
| `-k restricted` | `1 failed, 4 deselected`; the assertion lists the six packs |
| Quotes found | 37 of 37 found, rate 1.000 (threshold 0.95) |
| All five | `2 failed, 3 passed`: the access test (six packs) and the regressions (G02, *Insufficient evidence.*) |

---

## Part F: Fix the change and pass the gate (self-study)

Part F fixes the cause and runs the gate again.

The fix replaces the change's filter with `can_read`, the allow-list from Part B. Run the release set and the gate:

In [41]:
# fixed: the change with can_read as its filter, saved as results/fixed.json; then the gate on that file
fixed = run("fixed", can_read, "allow-list per user, k=3")
gate(RESULTS / "fixed.json")

fixed     passed 38 of 42 | saved results/fixed.json


.....                                                                    [100%]
5 passed in 0.02s



The fixed change passes 38 of 42, and the gate says `5 passed`: no pack holds a document its user may not read, and
G02 passes again. Compare the fixed change with today's release:

In [42]:
# change: today's release against the fixed change, question by question
change = paired([r["passed"] for r in baseline], [r["passed"] for r in fixed])
print("gained:", [baseline[i]["question_id"] for i in change["gained"]])
print("lost:  ", [(baseline[i]["question_id"], fixed[i]["answer"]) for i in change["lost"]])

gained: ['A01', 'A02', 'A03']
lost:   []


Gained A01, A02 and A03, lost nothing. A passing gate says that nothing the release forbids happened on these
questions. Whether the change is worth shipping is the paired comparison's call: here the agents gain and the customers
lose nothing.

The gate says no more than its five tests read:

- `test_quotes_found` checks that a quote's words are in the line it cites, not that the line means what the claim
  says (Lab 07's judge passed answers that quoted real lines).
- Its 0.95 lets about one quote in twenty go unfound (Part E), and a found quote may still not support its claim.
- The tests read the 42 saved questions: a leak or a lost answer on a question outside the release set is caught only
  once that question is added to it.

The same file runs anywhere the results files are, with no key, which is what a CI job needs. From the repository
folder:

```bash
GATE_BASELINE=path/to/baseline.json GATE_CANDIDATE=path/to/candidate.json python -m pytest 11_Release_Gate/test_release.py -q
```

The capstone runs it on its own service's results.

**Lesson to Remember**
> Fix the cause the gate names, not the gate. A passing gate says the rules held on the release set; the paired
> comparison says whether the change is worth shipping.

### Checkpoint F

| You should see | Our run |
|---|---|
| The fixed change | passed 38 of 42; `results/fixed.json` saved |
| The gate | `5 passed` |
| Paired with today's release | gained A01, A02, A03; lost none |

---

## What you can do now

- Label every document with who may read it, and list what each user may read with an allow-list.
- Make one filter per user with `can_read(user)`, a function that returns a function (a closure), and use it after the
  search (a post-filter, which can leave a customer no evidence) and inside it (a pre-filter).
- Check a permission filter on a question it should let through for one user and stop for another, and read the
  replies as well as the hits.
- Find a stale version with a version check and remove it by ID with `delete`, `add_documents` and `get_by_ids`.
- Save one row per question for a release and for a change, with a build manifest.
- Write release tests with pytest, run them on saved results with no key, read a failing assertion back to its rows,
  say what the quotes-found threshold allows, and explain why a change with a higher average still fails on one leaked
  document.

## Going further: let the index remove old versions

Part C found the April document with a version check and deleted it by hand. LangChain's indexing API does this as
part of every update, which is called **incremental indexing**
([LangChain: indexing](https://reference.langchain.com/python/langchain-core/indexing/api/index)): a **record
manager** remembers which documents came from which source, and with `cleanup="incremental"` each new version of a
source deletes the old one. Index the five policies as they were in April, then as they are today:

In [43]:
# policy_store and manager: a new store and its record manager; the five policies indexed as in April, then as today
from langchain_core.indexing import InMemoryRecordManager, index
from langchain_core.vectorstores import InMemoryVectorStore
from shared.settings import lc_embeddings

policy_store = InMemoryVectorStore(lc_embeddings(settings))
manager = InMemoryRecordManager("atlas-policies")
manager.create_schema()
october = lc_pipeline.to_documents([{**policy, "chunk_id": policy["source_id"]} for policy in policies])
april_docs = lc_pipeline.to_documents([april]) + [doc for doc in october if doc.id != "H02"]
for name, documents in [("April", april_docs), ("October", october)]:
    print(name, index(documents, manager, policy_store, cleanup="incremental", source_id_key="source_id",
                      key_encoder="sha256"))
print(sorted((doc["metadata"]["source_id"], doc["metadata"]["version"]) for doc in policy_store.store.values()))

April {'num_added': 5, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}
October {'num_added': 1, 'num_updated': 0, 'num_skipped': 4, 'num_deleted': 1}
[('H01', '2026-10-01'), ('H02', '2026-10-01'), ('H03', '2026-10-01'), ('H04', '2026-10-01'), ('H05', '2026-10-01')]


### What happens step by step

1. `InMemoryRecordManager` remembers, for each document it indexed, its source and a hash of its content;
   `create_schema()` sets up its table.
2. `october` holds the five policies as documents; `april_docs` the same set with the April H02 in place of today's.
3. `index()` adds the documents whose hash it has not seen, skips the ones it has, and with `cleanup="incremental"`
   deletes the older documents of every source it just updated (`source_id_key` names the metadata field that holds
   the source).
4. The last line lists the source and version of everything the store holds.

The April run added five documents. The October run added the new H02, skipped the four policies that had not changed
and deleted the April H02 (`num_deleted: 1`), so each source keeps its current version. The record manager keeps its
own IDs (hashes), so the store's IDs are no longer `H01` to `H05`.

## Challenge

Use only what this lab showed. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Ship the fixed change with April's H02 left in.** Part C deleted the April document by hand. What does the gate
   say when nobody does?

   - Under **Task 1**, add the April document back, run the release set with `can_read` as `"april"`, run `gate()` on
     its file, and delete the April document again.

   **Check:** pytest names the version test and the April document's ID. Our worked solution passed 36 of 42 and got `2 failed, 3 passed`: the version test named `H02-2026-04-01 (H02 2026-04-01, current 2026-10-01)`, and the regression test named G01 (*Insufficient evidence.*) and G02 (*... stops working after 24 hours.*).

   **Explain:** which of the failing tests would still fail if the April document never reached a pack?

2. **Optional: check the citations instead of the packs.** Would a test that reads the cited IDs have caught the
   change in Part D?

   - Complete `cited_unreadable()` under **Task 2**: the `(question_id, source_id)` of every claim whose source the
     row's user may not read. Run it on `candidate` and on `fixed`.

   **Check:** two lists. Our worked solution found none in either: `candidate: []` and `fixed: []`.

   **Explain:** what does a citation check read, and what did the model read?

**Task 1.** Complete the cell, then run it:

In [44]:
# TODO: add the April document back with store.add_documents(lc_pipeline.to_documents([april])),
#       run the release set: run("april", can_read, "allow-list per user, k=3, April H02 left in"),
#       run the gate on it: gate(RESULTS / "april.json"),
#       then delete the April document again with store.delete(["H02-2026-04-01"])
print("outdated after the delete:", outdated(store))

outdated after the delete: []


**Task 2 (optional).** Complete `cited_unreadable()`, then run the cell:

In [45]:
# Task 2: cited_unreadable(rows) lists the claims whose cited source the row's user may not read
def cited_unreadable(rows):
    """(question_id, source_id) for every claim whose source the row's user may not read."""
    # TODO: loop over the rows and their claims; keep a claim when access.get(claim["source_id"], "unknown")
    #       is not in USERS[row["user"]]
    return None

print("candidate:", cited_unreadable(candidate))
print("fixed:    ", cited_unreadable(fixed))

candidate: None
fixed:     None


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `KeyError: 'access'` in `can_read` | the store was built from `load_corpus()` without Part A's labels | Run Part A's `records` cell and build the store from `records` |
| `FileNotFoundError: [Errno 2] No such file or directory: '.../results/candidate.json'` in pytest's output | the results file was not saved yet, or `GATE_CANDIDATE` names a path that does not exist | Run Part D first, or check the path you set |
| `ERROR: file or directory not found: 11_Release_Gate/test_release.py` | pytest was started from inside the lab folder with the path from the repository folder | Start it from the repository folder, or run `pytest test_release.py` inside the lab folder |

## Resources

- [LangChain: vector stores](https://docs.langchain.com/oss/python/integrations/vectorstores): `add_documents` with
  IDs, `delete`, `get_by_ids` and metadata filters
- [LangChain: indexing API](https://reference.langchain.com/python/langchain-core/indexing/api/index): the record manager and
  `cleanup` modes that remove old versions
- [pytest: get started](https://docs.pytest.org/en/stable/getting-started.html) and
  [fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html): test discovery, `assert` messages and fixtures
- [Python: subprocess.run](https://docs.python.org/3/library/subprocess.html#subprocess.run): how `gate()` starts
  pytest in a new process
- [OWASP: LLM02 sensitive information disclosure](https://genai.owasp.org/llmrisk/llm022025-sensitive-information-disclosure/):
  why access control belongs before the model, not in the prompt

Next: [the capstone](../Capstone_RAG_Service/README.md), *How to ship a RAG service with its release gate*. Back to [the lab list](../README.md).